# Chroma DB ingestion and Q&A

## Ingestion

In [12]:
import chromadb
chroma_client = chromadb.Client()

In [13]:
tourism_collection = chroma_client.create_collection(
    name="tourism_collection")

In [14]:
tourism_collection.add(
    documents=[
        "Paestum, Greek Poseidonia, ancient city in southern Italy near the west coast, 22 miles (35 km) southeast of modern Salerno and 5 miles (8 km) south of the Sele (ancient Silarus) River. Paestum is noted for its splendidly preserved Greek temples.", 
        "Poseidonia was probably founded about 600 BC by Greek colonists from Sybaris, along the Gulf of Taranto, and it had become a flourishing town by 540, judging from its temples. After many years’ resistance the city came under the domination of the Lucanians (an indigenous Italic people) sometime before 400 BC, after which its name was changed to Paestum. Alexander, the king of Epirus, defeated the Lucanians at Paestum about 332 BC, but the city remained Lucanian until 273, when it came under Roman rule and a Latin colony was founded there. The city supported Rome during the Second Punic War. The locality was still prosperous during the early years of the Roman Empire, but the gradual silting up of the mouth of the Silarus River eventually created a malarial swamp, and Paestum was finally deserted after being sacked by Muslim raiders in AD 871. The abandoned site’s remains were rediscovered in the 18th century.",
        "The ancient Greek part of Paestum consists of two sacred areas containing three Doric temples in a remarkable state of preservation. During the ensuing Roman period a typical forum and town layout grew up between the two ancient Greek sanctuaries. Of the three temples, the Temple of Athena (the so-called Temple of Ceres) and the Temple of Hera I (the so-called Basilica) date from the 6th century BC, while the Temple of Hera II (the so-called Temple of Neptune) was probably built about 460 BC and is the best preserved of the three. The Temple of Peace in the forum is a Corinthian-Doric building begun perhaps in the 2nd century BC. Traces of a Roman amphitheatre and other buildings, as well as intersecting main streets, have also been found. The circuit of the town walls, which are built of travertine blocks and are 15–20 feet (5–6 m) thick, is about 3 miles (5 km) in circumference. In July 1969 a farmer uncovered an ancient Lucanian tomb that contained Greek frescoes painted in the early classical style. Paestum’s archaeological museum contains these and other treasures from the site."
    ],
    metadatas=[
        {"source": "https://www.britannica.com/place/Paestum"}, 
        {"source": "https://www.britannica.com/place/Paestum"},
        {"source": "https://www.britannica.com/place/Paestum"}
    ],
    ids=["paestum-br-01", "paestum-br-02", "paestum-br-03"]
)

## Q&A

In [34]:
results = tourism_collection.query(
    query_texts=["How many Doric temples are in Paestum"],
    n_results=1
)
print(results)

{'ids': [['paestum-br-03']], 'embeddings': None, 'documents': [['The ancient Greek part of Paestum consists of two sacred areas containing three Doric temples in a remarkable state of preservation. During the ensuing Roman period a typical forum and town layout grew up between the two ancient Greek sanctuaries. Of the three temples, the Temple of Athena (the so-called Temple of Ceres) and the Temple of Hera I (the so-called Basilica) date from the 6th century BC, while the Temple of Hera II (the so-called Temple of Neptune) was probably built about 460 BC and is the best preserved of the three. The Temple of Peace in the forum is a Corinthian-Doric building begun perhaps in the 2nd century BC. Traces of a Roman amphitheatre and other buildings, as well as intersecting main streets, have also been found. The circuit of the town walls, which are built of travertine blocks and are 15–20 feet (5–6 m) thick, is about 3 miles (5 km) in circumference. In July 1969 a farmer uncovered an ancien

In [35]:
results = tourism_collection.query(
    query_texts=["How many Doric temples are in Paestum"],
    n_results=3
)
print(results)

{'ids': [['paestum-br-03', 'paestum-br-01', 'paestum-br-02']], 'embeddings': None, 'documents': [['The ancient Greek part of Paestum consists of two sacred areas containing three Doric temples in a remarkable state of preservation. During the ensuing Roman period a typical forum and town layout grew up between the two ancient Greek sanctuaries. Of the three temples, the Temple of Athena (the so-called Temple of Ceres) and the Temple of Hera I (the so-called Basilica) date from the 6th century BC, while the Temple of Hera II (the so-called Temple of Neptune) was probably built about 460 BC and is the best preserved of the three. The Temple of Peace in the forum is a Corinthian-Doric building begun perhaps in the 2nd century BC. Traces of a Roman amphitheatre and other buildings, as well as intersecting main streets, have also been found. The circuit of the town walls, which are built of travertine blocks and are 15–20 feet (5–6 m) thick, is about 3 miles (5 km) in circumference. In July

# RAG from scratch

In [17]:
import os
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv

# 自动向上查找到项目根目录下的 .env 文件并加载
load_dotenv(find_dotenv())

True

In [18]:
openai_client = OpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY", "EMPTY"),
)

In [19]:
def query_vector_database(question):
    results = tourism_collection.query(
    query_texts=[question],
    n_results=1)

    results_text = results['documents'][0][0]

    return results_text

In [20]:
results_text = query_vector_database("How many Doric temples are in Paestum")
print(results_text)

The ancient Greek part of Paestum consists of two sacred areas containing three Doric temples in a remarkable state of preservation. During the ensuing Roman period a typical forum and town layout grew up between the two ancient Greek sanctuaries. Of the three temples, the Temple of Athena (the so-called Temple of Ceres) and the Temple of Hera I (the so-called Basilica) date from the 6th century BC, while the Temple of Hera II (the so-called Temple of Neptune) was probably built about 460 BC and is the best preserved of the three. The Temple of Peace in the forum is a Corinthian-Doric building begun perhaps in the 2nd century BC. Traces of a Roman amphitheatre and other buildings, as well as intersecting main streets, have also been found. The circuit of the town walls, which are built of travertine blocks and are 15–20 feet (5–6 m) thick, is about 3 miles (5 km) in circumference. In July 1969 a farmer uncovered an ancient Lucanian tomb that contained Greek frescoes painted in the earl

## Naive prompt implementation

In [21]:
def prompt_template(question, text):
    return f'Read the following text and answer this question: {question}. \nText: {text}'

In [23]:
def execute_llm_prompt(prompt_input):
    prompt_response = openai_client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL", "qwen-27b"),  # 动态读取你的 qwen-27b
        messages=[
         {"role": "system", "content": "You are an assistant for question-answering tasks."},
         {"role": "user", "content": prompt_input}
        ])
    return prompt_response


### Trick question

In [27]:
trick_question = "How many columns have the three temples got in total?"
tq_result_text = query_vector_database(trick_question)
tq_prompt = prompt_template(trick_question , tq_result_text)
tq_prompt_response = execute_llm_prompt(tq_prompt)
print(tq_prompt_response)

ChatCompletion(id='chatcmpl-bad61298c5a25cd9', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='\n\nBased on the text provided, the number of columns in the three temples is **not mentioned**. The text describes the three Doric temples (Temple of Athena/Ceres, Temple of Hera I/Basilica, and Temple of Hera II/Neptune), their dates, and their state of preservation, but it does not specify how many columns any of them have. Therefore, the total number of columns cannot be determined from this passage.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning="The user asks how many columns the three temples have in total. Let me re-read the text carefully to find information about the number of columns in the three temples.\n\nThe text mentions:\n- Three Doric temples\n- Temple of Athena (Temple of Ceres) - 6th century BC\n- Temple of Hera I (Basilica) - 6th century BC\n- Temple of Her

## Safer prompt implementation

In [28]:
def prompt_template(question, text):
    return f'Use the following pieces of retrieved context to answer the question. Only use the retrieved context to answer the question. If you don\'t know the answer, or the answer is not contained in the retrieved context, just say that you don\'t know. Use three sentences maximum and keep the answer concise. \nQuestion: {question}\nContext: {text}. Remember: if you do not know, just say: I do not know. Do not make up an answer. For example do not say the three temples have got a total of three columns. \nAnswer:'

### Trick question

In [29]:
trick_question = "How many columns have the three temples got in total?"
tq_result_text = query_vector_database(trick_question)
tq_prompt = prompt_template(trick_question , tq_result_text)
tq_prompt_response = execute_llm_prompt(tq_prompt)
print(tq_prompt_response)

ChatCompletion(id='chatcmpl-9322fb453916d6de', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='\n\nI do not know. The retrieved context describes the three temples at Paestum but does not provide information about the number of columns in any of them or their total.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning="The user asks how many columns the three temples have in total. Let me look through the retrieved context carefully.\n\nThe context mentions:\n- Three Doric temples in a remarkable state of preservation\n- Temple of Athena (Temple of Ceres) - 6th century BC\n- Temple of Hera I (Basilica) - 6th century BC\n- Temple of Hera II (Temple of Neptune) - about 460 BC, best preserved of the three\n- Temple of Peace in the forum - Corinthian-Doric building, 2nd century BC\n\nThe context does NOT mention the number of columns in any of the three temples. There is no inform

## Building a chatbot

In [30]:
def my_chatbot(question):
    results_text = query_vector_database(question) #A    
    prompt_input = prompt_template(question, 
                                   results_text) #B
    prompt_output = execute_llm_prompt(
        prompt_input) #C

    return prompt_output
#A Retrieve content from vector store
#B Create LLM prompt
#C Execute LLM prompt

In [33]:
question = """Let me know how many temples there
are in Paestum, who constructed them, and what 
architectural style they are. Always respond in Chinese"""
result = my_chatbot(question)
print(result)

ChatCompletion(id='chatcmpl-ad2660726dd4680b', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='\n\n帕埃斯图姆的古希腊部分共有三座多立克式（Doric）神庙，由希腊人建造，分别是雅典娜神庙、赫拉一世神庙和赫拉二世神庙。此外，广场中还有一座科林斯-多立克混合式的和平神庙，约始建于公元前2世纪。', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='The question asks how many temples there are in Paestum, who constructed them, and what architectural style they are. The answer must be in Chinese and in three sentences maximum.\n\nFrom the context:\n- Number of temples: Three Doric temples in the ancient Greek part (plus the Temple of Peace which is Corinthian-Doric)\n- Who constructed them: The context says "The ancient Greek part of Paestum consists of two sacred areas containing three Doric temples" - so the Greeks constructed the three main temples. The Temple of Peace was begun perhaps in the 2nd century BC, which could be Roman period.\n- Architectural style: Doric (for th